# Hands On UA 3 e UA 4: Implementação do QAOA para o Problema do Caixeiro Viajante

**Integrantes:** Enderson Luiz Pereira Junior e Gabriel Augusto David
**Disciplina:** Algoritmos Quânticos
**Atividade:** Implementação do QAOA para o Problema do Caixeiro Viajante

## 1. Introdução

O Problema do Caixeiro Viajante, conhecido como *Travelling Salesman Problem* (TSP), é um problema clássico de otimização combinatória. Seu objetivo é encontrar a rota de menor custo que visita todas as cidades exatamente uma vez e retorna à cidade inicial.

Em termos de teoria dos grafos, cada cidade é representada por um vértice, enquanto as conexões entre as cidades são representadas por arestas ponderadas. Os pesos das arestas podem representar distâncias, tempos de deslocamento, custos financeiros ou qualquer outra métrica relevante para o problema.

Embora seu enunciado seja simples, o TSP apresenta crescimento combinatório. Para um conjunto com $n$ cidades, existem muitas ordens diferentes em que essas cidades podem ser visitadas. Quando a cidade inicial é fixada, o número de rotas possíveis é dado por:

$$
(n-1)!
$$

Se rotas percorridas em sentidos opostos forem consideradas equivalentes em um grafo simétrico, esse número pode ser reduzido para:

$$
\frac{(n-1)!}{2}
$$

Mesmo com essa redução, o número de possibilidades cresce rapidamente à medida que novas cidades são adicionadas. Por essa razão, métodos de busca exaustiva tornam-se pouco viáveis para instâncias grandes, embora ainda sejam adequados para os casos pequenos avaliados neste trabalho.

O *Quantum Approximate Optimization Algorithm* (QAOA) é um algoritmo híbrido clássico-quântico voltado à obtenção de soluções aproximadas para problemas de otimização combinatória. Sua execução combina um circuito quântico parametrizado com um otimizador clássico. A parte quântica prepara e avalia estados candidatos, enquanto a parte clássica ajusta os parâmetros do circuito com o objetivo de minimizar o valor esperado de um Hamiltoniano de custo.

Neste notebook, serão implementadas duas abordagens para o TSP:

1. uma solução clássica exata, baseada na avaliação das rotas possíveis;
2. uma solução aproximada baseada no QAOA.

As duas abordagens serão aplicadas a instâncias com $3$, $4$, $5$ e $6$ cidades. Os resultados serão comparados considerando as rotas encontradas, os custos associados, os tempos de execução e os recursos necessários para a representação e execução dos circuitos quânticos.

### 1.1 Objetivos

O objetivo geral deste trabalho é implementar e analisar o QAOA aplicado ao Problema do Caixeiro Viajante, comparando seus resultados com uma solução clássica exata em instâncias pequenas.

Como objetivos específicos, busca-se:

- compreender os fundamentos do Problema do Caixeiro Viajante;
- representar as instâncias do TSP por meio de grafos e matrizes de distâncias;
- implementar uma solução clássica exata para determinar a rota ótima;
- formular o TSP por meio de variáveis binárias e restrições;
- construir o Hamiltoniano de custo utilizado pelo QAOA;
- compreender o papel do Hamiltoniano misturador;
- implementar o circuito parametrizado do QAOA;
- otimizar os parâmetros do circuito por meio de um algoritmo clássico;
- executar as abordagens clássica e quântica para instâncias de 3, 4, 5 e 6 cidades;
- reconstruir e validar as rotas obtidas;
- comparar os custos das rotas e os tempos de execução;
- registrar os circuitos, histogramas e demais resultados das execuções;
- discutir as vantagens, limitações e perspectivas práticas do QAOA.

### 1.2 Organização dos experimentos

Os experimentos serão organizados em quatro cenários, definidos pela quantidade de cidades:

- **Cenário 1:** 3 cidades;
- **Cenário 2:** 4 cidades;
- **Cenário 3:** 5 cidades;
- **Cenário 4:** 6 cidades.

Cada cenário será representado por uma matriz de distâncias simétrica. O elemento $d_{ij}$ da matriz representa o custo de deslocamento da cidade $i$ para a cidade $j$.

A mesma matriz será utilizada nas abordagens clássica e quântica, garantindo que os métodos sejam avaliados sobre dados de entrada equivalentes.

A solução clássica exata será utilizada como referência para determinar:

- a rota ótima;
- o menor custo possível;
- o número de rotas avaliadas;
- o tempo necessário para encontrar a solução.

Na abordagem quântica, serão registrados:

- a quantidade de variáveis binárias e qubits;
- o Hamiltoniano de custo;
- o número de camadas do QAOA;
- o otimizador clássico utilizado;
- os parâmetros encontrados;
- a rota reconstruída;
- o custo da solução;
- a diferença relativa em relação ao ótimo clássico;
- o tempo total de execução;
- a profundidade e a quantidade de operações do circuito;
- a distribuição dos resultados de medição.

A diferença relativa entre o custo obtido pelo QAOA e o custo ótimo clássico será calculada por:

$$
\text{diferença relativa}
=
\frac{C_{\text{QAOA}} - C_{\text{ótimo}}}
{C_{\text{ótimo}}}
$$

Quando o QAOA encontrar uma rota com o mesmo custo da solução clássica, a diferença relativa será igual a zero.

Além dos valores numéricos, serão apresentados grafos das instâncias, representações das rotas encontradas, circuitos quânticos, histogramas e tabelas comparativas. Esses resultados serão utilizados para discutir o desempenho e as limitações práticas do QAOA.

## 2. Fundamentos do Problema do Caixeiro Viajante

O Problema do Caixeiro Viajante, conhecido como *Travelling Salesman Problem* (TSP), consiste em determinar a rota de menor custo que visita todas as cidades exatamente uma vez e retorna à cidade de origem.

O TSP pertence à classe dos problemas de otimização combinatória. Uma solução possível é definida pela ordem em que as cidades são visitadas, enquanto a qualidade dessa solução é determinada pela soma dos custos dos deslocamentos realizados.

O problema pode representar diferentes situações práticas. Os vértices podem corresponder a cidades, pontos de entrega, centros de distribuição, máquinas ou tarefas. Os pesos das conexões podem representar distância, tempo, consumo de energia, custo financeiro ou outra medida que se deseja minimizar.

### 2.1 Representação por grafos

Uma instância do TSP pode ser representada por um grafo ponderado:

$$
G = (V, E)
$$

em que:

- $V$ representa o conjunto de vértices, correspondentes às cidades;
- $E$ representa o conjunto de arestas, correspondentes às conexões entre as cidades;
- cada aresta possui um peso associado ao custo do deslocamento.

Se existem $n$ cidades, o conjunto de vértices pode ser escrito como:

$$
V = \{0, 1, 2, \ldots, n-1\}
$$

Os custos das conexões são organizados em uma matriz de distâncias:

$$
D =
\begin{bmatrix}
d_{00} & d_{01} & \cdots & d_{0,n-1} \\
d_{10} & d_{11} & \cdots & d_{1,n-1} \\
\vdots & \vdots & \ddots & \vdots \\
d_{n-1,0} & d_{n-1,1} & \cdots & d_{n-1,n-1}
\end{bmatrix}
$$

O elemento $d_{ij}$ representa o custo de viajar da cidade $i$ para a cidade $j$.

Nas instâncias simétricas utilizadas neste trabalho, considera-se que:

$$
d_{ij} = d_{ji}
$$

Isso significa que o custo de viajar da cidade $i$ para a cidade $j$ é igual ao custo do percurso inverso.

Além disso, os elementos da diagonal principal são iguais a zero:

$$
d_{ii} = 0
$$

pois não há custo de deslocamento de uma cidade para ela mesma.

Uma solução válida do TSP corresponde a um ciclo hamiltoniano, isto é, um ciclo que visita todos os vértices exatamente uma vez e retorna ao vértice inicial.

### 2.2 Crescimento do espaço de busca

Uma rota do TSP pode ser representada por uma permutação das cidades. Para $n$ cidades, existem inicialmente $n!$ ordenações possíveis. Entretanto, como a rota é cíclica, pode-se fixar uma cidade como ponto inicial sem perder soluções distintas. Neste trabalho, a cidade $0$ será utilizada como origem. Com essa redução, o número de rotas que precisam ser avaliadas passa a ser $(n-1)!$. Para grafos simétricos, uma rota e sua ordem inversa possuem o mesmo custo. Por exemplo:

$$
0 \rightarrow 1 \rightarrow 2 \rightarrow 3 \rightarrow 0
$$

e:

$$
0 \rightarrow 3 \rightarrow 2 \rightarrow 1 \rightarrow 0
$$

percorrem as mesmas arestas em sentidos opostos. Caso essas rotas sejam consideradas equivalentes, o número de ciclos distintos pode ser expresso como:

$$
\frac{(n-1)!}{2}
$$

Para as instâncias avaliadas neste trabalho, o crescimento do espaço de busca pode ser observado a seguir:

| Número de cidades | Permutações com origem fixa | Rotas distintas em grafo simétrico |
|---:|---:|---:|
| 3 | $2! = 2$ | $1$ |
| 4 | $3! = 6$ | $3$ |
| 5 | $4! = 24$ | $12$ |
| 6 | $5! = 120$ | $60$ |

Embora esses valores ainda sejam pequenos e permitam uma solução exata por enumeração, o crescimento fatorial torna a busca exaustiva rapidamente inviável para instâncias maiores.

### 2.3 Função de custo da rota

Considere uma rota representada pela sequência:

$$
\pi = (\pi_0, \pi_1, \ldots, \pi_{n-1})
$$

em que cada elemento $\pi_p$ indica a cidade visitada na posição $p$.

O custo total da rota é obtido pela soma dos custos entre cidades consecutivas, incluindo o retorno da última cidade para a cidade inicial:

$$
C(\pi)
=
\sum_{p=0}^{n-2}
d_{\pi_p,\pi_{p+1}}
+
d_{\pi_{n-1},\pi_0}
$$

De forma equivalente, utilizando índices de maneira cíclica:

$$
C(\pi)
=
\sum_{p=0}^{n-1}
d_{\pi_p,\pi_{(p+1)\bmod n}}
$$

O objetivo do TSP é encontrar a rota $\pi^*$ que apresenta o menor custo:

$$
\pi^*
=
\operatorname*{arg\,min}_{\pi}
C(\pi)
$$

A solução clássica exata avaliará as rotas possíveis e selecionará aquela com menor custo. Esse valor será utilizado como referência para analisar a qualidade das soluções produzidas pelo QAOA.

## 3. Solução clássica exata

A solução clássica utilizada neste trabalho será baseada na enumeração das rotas possíveis. Como as instâncias possuem entre $3$ e $6$ cidades, é possível avaliar todas as rotas distintas e determinar com certeza aquela que apresenta o menor custo.

Esse procedimento é considerado exato porque não depende de aproximações ou heurísticas. Todas as soluções possíveis, após a aplicação das reduções de simetria, são analisadas. Dessa forma, a menor rota encontrada corresponde ao ótimo global da instância.

A solução clássica terá duas funções principais no experimento:

1. determinar a rota ótima e seu respectivo custo;
2. fornecer uma referência para avaliar a qualidade das soluções obtidas pelo QAOA.

Embora a enumeração completa seja adequada para as instâncias pequenas deste trabalho, seu custo cresce de forma fatorial. Por isso, esse método se torna rapidamente inviável quando o número de cidades aumenta.

### 3.1 Busca por permutações

Uma rota pode ser representada como uma permutação das cidades. Para evitar que rotas equivalentes sejam geradas apenas por mudanças no ponto de partida, a cidade $0$ será fixada como origem.

Assim, em vez de gerar permutações de todas as $n$ cidades, serão geradas apenas as permutações das cidades restantes:

$$
\{1,2,\ldots,n-1\}
$$

Para cada permutação $\sigma$, será construída uma rota da forma:

$$
\pi = (0,\sigma_1,\sigma_2,\ldots,\sigma_{n-1})
$$

O retorno à cidade inicial será considerado no cálculo do custo, sem a necessidade de incluir novamente a cidade $0$ na representação interna da rota.

Por exemplo, para $4$ cidades, uma permutação possível é:

$$
(1,3,2)
$$

A rota correspondente é:

$$
0 \rightarrow 1 \rightarrow 3 \rightarrow 2 \rightarrow 0
$$

Como a cidade inicial é fixada, o número inicial de permutações avaliadas é:

$$
(n-1)!
$$

O procedimento clássico pode ser descrito pelo seguinte pseudocódigo:

1. fixar a cidade $0$ como origem;
2. gerar as permutações das demais cidades;
3. construir uma rota para cada permutação;
4. calcular o custo total de cada rota;
5. armazenar a rota de menor custo;
6. retornar a melhor rota e seu custo.

### 3.2 Redução de simetria

Nas matrizes simétricas utilizadas neste trabalho, o custo de viajar da cidade $i$ para a cidade $j$ é igual ao custo do percurso inverso:

$$
d_{ij}=d_{ji}
$$

Consequentemente, uma rota e sua ordem inversa possuem o mesmo custo. Por exemplo:

$$
0 \rightarrow 1 \rightarrow 2 \rightarrow 3 \rightarrow 0
$$

e:

$$
0 \rightarrow 3 \rightarrow 2 \rightarrow 1 \rightarrow 0
$$

representam o mesmo ciclo percorrido em sentidos opostos.

Para evitar a avaliação duplicada dessas soluções, será utilizada uma condição adicional. Entre uma rota e sua inversa, apenas uma delas será considerada. Na implementação, isso pode ser feito mantendo somente as rotas em que a primeira cidade visitada após a origem possui índice menor que a última cidade visitada antes do retorno:

$$
\pi_1 < \pi_{n-1}
$$

Com essa redução, o número de rotas distintas avaliadas passa de:

$$
(n-1)!
$$

para:

$$
\frac{(n-1)!}{2}
$$

para instâncias simétricas com mais de duas cidades.

Essa redução não altera o resultado ótimo, pois apenas elimina representações equivalentes da mesma rota. Ela reduz a quantidade de avaliações necessárias e torna a busca clássica mais eficiente.

### 3.3 Cálculo do custo da rota

Para cada rota gerada, o custo total é calculado pela soma dos pesos das arestas percorridas.

Considere uma rota:

$$
\pi=(\pi_0,\pi_1,\ldots,\pi_{n-1})
$$

O custo é dado por:

$$
C(\pi)
=
\sum_{p=0}^{n-2}
d_{\pi_p,\pi_{p+1}}
+
d_{\pi_{n-1},\pi_0}
$$

O primeiro termo representa o custo dos deslocamentos entre cidades consecutivas. O segundo termo representa o retorno da última cidade para a cidade inicial.

Como exemplo, considere a rota:

$$
0 \rightarrow 1 \rightarrow 3 \rightarrow 2 \rightarrow 0
$$

Seu custo será:

$$
C(\pi)
=
d_{0,1}
+
d_{1,3}
+
d_{3,2}
+
d_{2,0}
$$

Durante a busca, esse valor será comparado com o menor custo encontrado até o momento. Sempre que uma rota de custo menor for identificada, ela passará a ser armazenada como a melhor solução.

Ao final da execução, a solução clássica retornará:

- a rota ótima;
- o custo total da rota;
- a quantidade de rotas avaliadas;
- o tempo de execução;
- as rotas equivalentes que também apresentam o custo ótimo, quando existirem.

A complexidade temporal do procedimento é aproximadamente:

$$
O\left((n-1)! \cdot n\right)
$$

pois são avaliadas até $(n-1)!$ permutações e o cálculo do custo de cada rota percorre $n$ deslocamentos. Mesmo com a redução de simetria, o crescimento permanece fatorial, limitando a aplicação desse método a instâncias pequenas.

## 4. Fundamentos do QAOA

O *Quantum Approximate Optimization Algorithm* (QAOA) é um algoritmo variacional híbrido desenvolvido para buscar soluções aproximadas de problemas de otimização combinatória. Sua execução combina um circuito quântico parametrizado com um método clássico de otimização.

A parte quântica prepara estados candidatos e estima o valor esperado de uma função de custo representada por um Hamiltoniano. A parte clássica recebe esse valor e ajusta os parâmetros do circuito, buscando reduzir progressivamente o custo da solução.

No contexto deste trabalho, o objetivo do QAOA é aumentar a probabilidade de medir bitstrings que representem rotas válidas e de baixo custo para o Problema do Caixeiro Viajante. A atividade solicita a definição do Hamiltoniano do TSP, a construção do circuito parametrizado e a otimização clássica de seus parâmetros para instâncias de diferentes tamanhos.

O funcionamento geral do QAOA pode ser representado pelas seguintes etapas:

1. preparar um estado quântico inicial;
2. aplicar uma operação associada ao custo do problema;
3. aplicar uma operação de mistura;
4. repetir essas operações por um número definido de camadas;
5. medir o circuito;
6. calcular o valor esperado do Hamiltoniano;
7. atualizar os parâmetros por meio de um otimizador clássico;
8. repetir o processo até que um critério de parada seja atingido.

### 4.1 Estrutura híbrida clássico-quântica

O QAOA é denominado híbrido porque distribui sua execução entre componentes quânticos e clássicos.

A etapa quântica é responsável por preparar e medir um estado parametrizado. A etapa clássica analisa o valor da função objetivo e escolhe novos parâmetros para a próxima execução do circuito.

O processo começa com um conjunto inicial de parâmetros:

$$
\boldsymbol{\gamma}
=
(\gamma_1,\gamma_2,\ldots,\gamma_p)
$$

e:

$$
\boldsymbol{\beta}
=
(\beta_1,\beta_2,\ldots,\beta_p)
$$

Esses parâmetros controlam, respectivamente, a aplicação do Hamiltoniano de custo e do Hamiltoniano misturador.

Para cada conjunto de parâmetros, o circuito prepara um estado quântico:

$$
|\psi_p(\boldsymbol{\gamma},\boldsymbol{\beta})\rangle
$$

Em seguida, calcula-se o valor esperado do Hamiltoniano de custo:

$$
F_p(\boldsymbol{\gamma},\boldsymbol{\beta})
=
\langle
\psi_p(\boldsymbol{\gamma},\boldsymbol{\beta})
|
H_C
|
\psi_p(\boldsymbol{\gamma},\boldsymbol{\beta})
\rangle
$$

Como o TSP é um problema de minimização, o otimizador clássico procura parâmetros que reduzam esse valor:

$$
(\boldsymbol{\gamma}^*,\boldsymbol{\beta}^*)
=
\operatorname*{arg\,min}_{\boldsymbol{\gamma},\boldsymbol{\beta}}
F_p(\boldsymbol{\gamma},\boldsymbol{\beta})
$$

Após a otimização, o circuito é executado com os melhores parâmetros encontrados. As bitstrings medidas são então interpretadas como possíveis soluções do problema.

O fluxo híbrido pode ser resumido da seguinte forma:

$$
\text{parâmetros}
\rightarrow
\text{circuito quântico}
\rightarrow
\text{medições}
\rightarrow
\text{valor de custo}
\rightarrow
\text{otimizador clássico}
\rightarrow
\text{novos parâmetros}
$$

Esse ciclo é repetido até que o número máximo de iterações seja atingido ou que a melhoria entre execuções se torne suficientemente pequena.

### 4.2 Hamiltoniano de custo

O Hamiltoniano de custo, representado por $H_C$, codifica a função objetivo do problema de otimização.

No TSP, os menores valores de energia devem estar associados a rotas válidas e de menor distância. Portanto, o Hamiltoniano precisa representar tanto o custo das arestas percorridas quanto as restrições necessárias para formar uma rota válida.

De forma geral, ele pode ser escrito como:

$$
H_C
=
H_{\text{distância}}
+
H_{\text{restrições}}
$$

O termo $H_{\text{distância}}$ atribui energia de acordo com o custo total da rota. Rotas mais longas ou mais caras recebem valores maiores.

O termo $H_{\text{restrições}}$ penaliza configurações inválidas, como:

- uma cidade aparecer mais de uma vez;
- uma cidade não aparecer na rota;
- duas cidades ocuparem a mesma posição;
- uma posição da rota permanecer vazia.

A operação quântica associada ao Hamiltoniano de custo é:

$$
U_C(\gamma)
=
e^{-i\gamma H_C}
$$

O parâmetro $\gamma$ determina por quanto tempo, ou com que intensidade, o estado quântico evolui sob a influência do Hamiltoniano de custo.

Essa operação introduz fases relativas nas amplitudes dos estados. Estados com diferentes custos acumulam diferentes fases, preparando o sistema para que a operação de mistura e a interferência quântica alterem posteriormente suas probabilidades de medição.

### 4.3 Hamiltoniano misturador

O Hamiltoniano misturador, representado por $H_M$, é responsável por promover transições entre diferentes estados candidatos do espaço de busca.

Uma escolha comum é o misturador baseado em portas de Pauli-X:

$$
H_M
=
\sum_{j=0}^{m-1} X_j
$$

em que $m$ é a quantidade de qubits e $X_j$ representa o operador de Pauli-X aplicado ao qubit $j$.

A operação de mistura é dada por:

$$
U_M(\beta)
=
e^{-i\beta H_M}
$$

Como os termos atuam individualmente sobre os qubits, essa operação pode ser implementada por rotações em torno do eixo $X$:

$$
U_M(\beta)
=
\prod_{j=0}^{m-1}
e^{-i\beta X_j}
$$

Na implementação com Qiskit, essas evoluções são normalmente decompostas em portas parametrizadas equivalentes.

O misturador impede que o estado permaneça limitado a uma única configuração. Ao alterar as amplitudes dos estados da base computacional, ele permite que o algoritmo explore diferentes soluções possíveis.

No caso do TSP, o misturador padrão pode gerar tanto configurações válidas quanto inválidas. Por essa razão, as restrições da rota são incorporadas ao Hamiltoniano de custo por meio de penalidades. Assim, configurações inválidas recebem energia elevada e se tornam menos favoráveis durante a otimização.

Existem misturadores alternativos capazes de preservar determinadas restrições do problema. Entretanto, neste trabalho será adotada uma formulação compatível com a implementação do Qiskit, mantendo as penalidades no Hamiltoniano de custo.

### 4.4 Parâmetros $\gamma$ e $\beta$

Os parâmetros $\gamma$ e $\beta$ controlam as duas operações fundamentais do QAOA.

O parâmetro $\gamma$ está associado ao Hamiltoniano de custo:

$$
U_C(\gamma)
=
e^{-i\gamma H_C}
$$

Ele controla as fases introduzidas de acordo com o custo de cada solução candidata.

O parâmetro $\beta$ está associado ao Hamiltoniano misturador:

$$
U_M(\beta)
=
e^{-i\beta H_M}
$$

Ele controla a redistribuição das amplitudes entre os estados da base computacional.

Para uma única camada do QAOA, o estado final pode ser escrito como:

$$
|\psi_1(\gamma_1,\beta_1)\rangle
=
U_M(\beta_1)
U_C(\gamma_1)
|\psi_0\rangle
$$

O estado inicial mais comum é a superposição uniforme:

$$
|\psi_0\rangle
=
|+\rangle^{\otimes m}
$$

em que:

$$
|+\rangle
=
\frac{|0\rangle+|1\rangle}{\sqrt{2}}
$$

Esse estado atribui inicialmente a mesma amplitude a todas as bitstrings possíveis.

Os valores de $\gamma$ e $\beta$ não são conhecidos antecipadamente. Eles são encontrados por um otimizador clássico, que executa repetidamente o circuito e utiliza o valor esperado do Hamiltoniano de custo para escolher novos parâmetros.

A qualidade do resultado depende, entre outros fatores, de:

- valores iniciais escolhidos;
- método clássico de otimização;
- número máximo de avaliações;
- profundidade do circuito;
- presença de mínimos locais;
- natureza probabilística das medições.

Por isso, diferentes execuções podem produzir parâmetros e soluções distintas, mesmo para a mesma instância.

### 4.5 Profundidade $p$

A profundidade $p$ do QAOA indica quantas vezes o par formado pelo Hamiltoniano de custo e pelo Hamiltoniano misturador é aplicado.

Para profundidade $p$, o estado preparado é:

$$
|\psi_p(\boldsymbol{\gamma},\boldsymbol{\beta})\rangle
=
U_M(\beta_p)
U_C(\gamma_p)
\cdots
U_M(\beta_2)
U_C(\gamma_2)
U_M(\beta_1)
U_C(\gamma_1)
|\psi_0\rangle
$$

Cada camada adiciona dois novos parâmetros:

$$
\gamma_l
\quad \text{e} \quad
\beta_l
$$

Portanto, um circuito com profundidade $p$ possui:

$$
2p
$$

parâmetros a serem otimizados.

Em princípio, valores maiores de $p$ permitem construir estados quânticos mais expressivos e podem aumentar a qualidade das soluções. Entretanto, o aumento da profundidade também produz:

- maior número de portas;
- circuitos mais profundos;
- maior tempo de simulação;
- otimização clássica mais difícil;
- maior sensibilidade a ruídos em hardware quântico;
- crescimento do número de parâmetros.

Assim, existe um compromisso entre qualidade da solução e custo de execução.

Neste trabalho, será utilizada inicialmente uma profundidade pequena, adequada às limitações do ambiente de simulação e às instâncias de 3 a 6 cidades. O valor adotado será registrado em cada execução, juntamente com a profundidade efetiva e a quantidade de operações do circuito.

Após a otimização, o circuito final será medido diversas vezes. As bitstrings mais frequentes serão interpretadas e verificadas para identificar quais representam rotas válidas e qual custo está associado a cada uma delas.

## 5. Formulação do TSP para o QAOA

Para resolver o Problema do Caixeiro Viajante com o QAOA, é necessário representar as rotas possíveis por estados quânticos e construir um Hamiltoniano que atribua menor energia às rotas de menor custo.

Neste trabalho, será utilizada uma codificação compacta por índice de rota. Inicialmente, são geradas todas as rotas válidas que começam na cidade $0$. Cada rota recebe um índice inteiro e esse índice é representado por uma bitstring.

Essa escolha evita a utilização da codificação one-hot tradicional, que exige $n^2$ variáveis binárias para $n$ cidades. Como os experimentos são limitados a instâncias de $3$ a $6$ cidades, a codificação por índice permite construir e simular circuitos menores, mantendo o foco na implementação e na análise do QAOA.

O processo de formulação será composto pelas seguintes etapas:

1. gerar as rotas válidas;
2. atribuir um índice binário a cada rota;
3. calcular o custo de cada rota;
4. atribuir penalidade aos índices que não representam rotas;
5. construir um Hamiltoniano diagonal;
6. converter esse Hamiltoniano para operadores de Pauli utilizados pelo Qiskit.

### 5.1 Variáveis binárias

A cidade $0$ será fixada como ponto inicial. As demais cidades serão organizadas por meio de permutações.

O conjunto de rotas representadas pode ser escrito como:

$$
\mathcal{R}_n
=
\left\{
(0,\sigma_1,\sigma_2,\ldots,\sigma_{n-1})
\mid
\sigma \in S_{n-1}
\right\}
$$

em que $S_{n-1}$ representa o conjunto de permutações das cidades:

$$
\{1,2,\ldots,n-1\}
$$

Como a cidade inicial está fixada, a quantidade de rotas representadas é:

$$
R=(n-1)!
$$

Cada rota recebe um índice:

$$
k \in \{0,1,\ldots,R-1\}
$$

Esse índice é representado por uma bitstring com $m$ bits, em que:

$$
m
=
\left\lceil
\log_2 R
\right\rceil
$$

Assim, cada estado da base computacional representa um índice:

$$
|z\rangle
=
|z_{m-1}z_{m-2}\ldots z_0\rangle
$$

O valor inteiro associado à bitstring é dado por:

$$
k(z)
=
\sum_{j=0}^{m-1}
2^jz_j
$$

Para as instâncias deste trabalho, a quantidade de rotas e de qubits será:

| Cidades | Rotas com origem fixa | Qubits necessários |
|---:|---:|---:|
| $3$ | $2!=2$ | $1$ |
| $4$ | $3!=6$ | $3$ |
| $5$ | $4!=24$ | $5$ |
| $6$ | $5!=120$ | $7$ |

Essa codificação permite representar as instâncias avaliadas com uma quantidade reduzida de qubits.

In [ ]:


6. Preparação do ambiente

7. Implementação modular

8. Cenário 1: 3 cidades

9. Cenário 2: 4 cidades

10. Cenário 3: 5 cidades

11. Cenário 4: 6 cidades

12. Comparação consolidada

13. Limitações, vantagens e perspectivas

14. Conclusão

15. Referências